# RFM 會員留存與流失分析 (Colab 腳本)
這個腳本是為了讓你能在需要時，分段執行並深入分析特定會員的 RFM（Recency, Frequency, Monetary）指標。
資料來源為專案目錄下的 `processed/betting_data.db`。

In [ ]:
import pandas as pd
import sqlite3
from pathlib import Path
import numpy as np

# 讀取資料
DB_PATH = "processed/betting_data.db"
conn = sqlite3.connect(DB_PATH)
df_raw = pd.read_sql("SELECT date, member_id, bet_amount FROM fact_daily_bets", conn)
conn.close()

df_raw["date"] = pd.to_datetime(df_raw["date"])
latest_date = df_raw["date"].max()
print(f"最新資料日期: {latest_date.strftime("%Y-%m-%d")}")
print(f"總投注筆數: {len(df_raw)}")

### 1. 會員基礎指標 (R, F, M 計算)
我們將計算每位會員的最後下注日、活躍天數、總投注額以及近期(30天內)投注額。

In [ ]:
# 計算基本 RFM 指標
df_raw["ym"] = df_raw["date"].dt.to_period("M")
monthly_stats = df_raw.groupby(["ym", "member_id"])["bet_amount"].sum().reset_index()

t_30 = latest_date - pd.Timedelta(days=30)
df_last30 = df_raw[df_raw["date"] > t_30]
last30_bet = df_last30.groupby("member_id")["bet_amount"].sum().to_dict()

member_stats = df_raw.groupby("member_id").agg(
    last_date=("date", "max"),
    total_active_days=("date", "nunique"),
    std_daily_bet=("bet_amount", "std"),
    total_months=("ym", "nunique"),
    historical_total_bet=("bet_amount", "sum")
).reset_index()

max_monthly = monthly_stats.groupby("member_id")["bet_amount"].max().to_dict()
member_stats.head()

### 2. 賽事偏好分析 (MLB vs NBA)
透過歷史下注月份，判斷該會員偏好 MLB 還是 NBA，這將用於區分「真流失」與「休賽季沉睡」。

In [ ]:
df_raw["month_num"] = df_raw["date"].dt.month
nba_months = [10, 11, 12, 1, 2, 3, 4, 5, 6]
mlb_months = [7, 8, 9]

seasonality = df_raw.groupby("member_id")["month_num"].apply(set).reset_index()
def get_season_tag(months_set):
    has_nba = any(m in nba_months for m in months_set)
    has_mlb = any(m in mlb_months for m in months_set)
    if has_nba and has_mlb: return "雙棲"
    elif has_mlb: return "純 MLB"
    elif has_nba: return "純 NBA"
    return "未知"

seasonality["賽事偏好"] = seasonality["month_num"].apply(get_season_tag)
season_dict = dict(zip(seasonality["member_id"], seasonality["賽事偏好"]))
seasonality["賽事偏好"].value_counts()

### 3. 貼上智能標籤
根據上述指標，給予每位會員「活躍中/流失/沉睡」以及「VIP/主力/一般」的標籤。

In [ ]:
current_month_num = latest_date.month

def get_current_profile(row):
    m_id = row["member_id"]
    days_since = (latest_date - row["last_date"]).days
    m_last30_bet = last30_bet.get(m_id, 0)
    max_bet = max_monthly.get(m_id, 0)
    season_tag = season_dict.get(m_id, "")

    if days_since <= 30:
        status = "🟢 活躍中"
        if m_last30_bet >= 40000: value = "⭐ VIP"
        elif m_last30_bet >= 12000: value = "🔥 主力"
        else: value = "👤 一般"
    else:
        is_sleeping = False
        if (current_month_num in mlb_months and "NBA" in season_tag) or \
           (current_month_num in nba_months and "MLB" in season_tag):
            is_sleeping = True

        if is_sleeping:
            status = "⚪ 沉睡中 (休賽季)"
        else:
            status = "🔴 已流失"

        if max_bet >= 40000: value = "⭐ 前 VIP"
        elif max_bet >= 12000: value = "🔥 前主力"
        else: value = "👤 前一般"

    days_per_month = row["total_active_days"] / row["total_months"]
    if days_per_month >= 12: freq_tag = "高頻玩家"
    elif days_per_month <= 4: freq_tag = "偶發玩家"
    else: freq_tag = "穩健玩家"

    std = row["std_daily_bet"] if pd.notna(row["std_daily_bet"]) else 0
    if std > 20000: vol_tag = "衝動大戶 (高波動)"
    elif std < 5000: vol_tag = "規律下注 (低波動)"
    else: vol_tag = "一般波動"

    return pd.Series([status, value, freq_tag, vol_tag, m_last30_bet, days_since])

member_stats[["當前狀態", "近期/歷史價值", "下注頻率", "下注波動度", "近30天總投注", "距今天數"]] = member_stats.apply(get_current_profile, axis=1)
member_stats["賽事偏好"] = member_stats["member_id"].map(season_dict)
member_stats.head()

### 4. 資料篩選與導出 (範例)
你可以修改下方的過濾條件，找出特定的目標客群，例如：找出「需挽回的大戶」。

In [ ]:
target_churn_vips = member_stats[
    (member_stats["當前狀態"] == "🔴 已流失") & 
    (member_stats["近期/歷史價值"].isin(["⭐ 前 VIP", "🔥 前主力"]))
]
print(f"共有 {len(target_churn_vips)} 位需挽回大戶。")
target_churn_vips.sort_values("historical_total_bet", ascending=False).head(10)